In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Read data

In [2]:
tmt_report = pd.read_csv('./article/CPTAC3_Clear_Cell_Renal_Cell_Carcinoma_Proteome.tmt10.tsv', sep='\t', index_col=0, skiprows=[1, 2, 3])
tmt_report = tmt_report.drop(['Authority', 'NCBIGeneID', 'Description', 'Organism', 'Chromosome', 'Locus'], axis=1)
# drop columns with " Unshared" in the name
tmt_report = tmt_report.loc[:, ~tmt_report.columns.str.contains(' Unshared')]
# remove "Log Ratio" from column names
tmt_report.columns = tmt_report.columns.str.replace(' Log Ratio', '')

In [3]:
metadata = pd.read_csv('./article/PDC_study_biospecimen_02242025_175220.tsv', sep='\t')
# drop all column except Tissue Type and Aliquot Submitter ID
metadata = metadata[['Aliquot Submitter ID', 'Tissue Type']]
# rename column Tissue Type to Condition, and Aliquot Submitter ID to Sample
metadata = metadata.rename(columns={'Tissue Type': 'Condition', 'Aliquot Submitter ID': 'Sample'})
# if Sample contains "QC" (case sensitive), add "QC" to Condition
metadata.loc[metadata['Sample'].str.contains('QC'), 'Condition'] = 'QC'
# drop rows with "Not Reported" in Condition
metadata = metadata[metadata['Condition'] != 'Not Reported']
metadata['Dataset'] = 'PDC000127'
# drop columns in tmt_report that are not in metadata
tmt_report = tmt_report.drop(set(tmt_report.columns) - set(metadata['Sample']), axis=1)

print(f"tmt_report shape: {tmt_report.shape}")
print(f"metadata shape: {metadata.shape}")

tmt_report shape: (9964, 202)
metadata shape: (202, 3)


In [5]:
# filter out QC samples
metadata = metadata[metadata['Condition'] != 'QC']
tmt_report = tmt_report[metadata['Sample']]

# save data
tmt_report.to_csv('report.csv', sep='\t')
metadata.to_csv('metadata.csv', sep='\t', index=False)